In [3]:
import pyqpanda3 as pq
import numpy as np

print("量子环境测试成功！")
print("当前 pyqpanda3 版本:", pq.__version__)

量子环境测试成功！
当前 pyqpanda3 版本: 0.3.4


In [4]:
"""Classical preprocessing for SPY price loading under the BSM model."""

from __future__ import annotations

import numpy as np


def lognormal_price_pdf(
    prices: np.ndarray,
    s0: float,
    r: float,
    sigma: float,
    maturity: float,
) -> np.ndarray:
    """Return the BSM log-normal density evaluated at the given prices."""
    if s0 <= 0:
        raise ValueError("s0 must be positive.")
    if sigma <= 0:
        raise ValueError("sigma must be positive.")
    if maturity <= 0:
        raise ValueError("maturity must be positive.")
    if np.any(prices <= 0):
        raise ValueError("All price grid points must be positive.")

    mu = np.log(s0) + (r - 0.5 * sigma**2) * maturity
    std = sigma * np.sqrt(maturity)

    log_term = (np.log(prices) - mu) / std
    normalizer = prices * std * np.sqrt(2.0 * np.pi)
    return np.exp(-0.5 * log_term**2) / normalizer


def discretize_bsm_price_distribution(
    s0: float = 500.0,
    r: float = 0.05,
    sigma: float = 0.15,
    maturity: float = 30.0 / 365.0,
    num_qubits: int = 3,
    truncation_sigma: float = 3.0,
) -> dict[str, np.ndarray | float | int]:
    """Discretize the BSM price distribution on a uniform price grid."""
    if num_qubits <= 0:
        raise ValueError("num_qubits must be positive.")
    if truncation_sigma <= 0:
        raise ValueError("truncation_sigma must be positive.")

    num_points = 2**num_qubits
    sqrt_t = np.sqrt(maturity)

    price_min = s0 * np.exp(-truncation_sigma * sigma * sqrt_t)
    price_max = s0 * np.exp(truncation_sigma * sigma * sqrt_t)
    price_grid = np.linspace(price_min, price_max, num_points)

    pdf_values = lognormal_price_pdf(
        prices=price_grid,
        s0=s0,
        r=r,
        sigma=sigma,
        maturity=maturity,
    )

    grid_spacing = price_grid[1] - price_grid[0]
    unnormalized_probabilities = pdf_values * grid_spacing
    probabilities = unnormalized_probabilities / np.sum(unnormalized_probabilities)
    amplitudes = np.sqrt(probabilities)

    return {
        "num_qubits": num_qubits,
        "num_points": num_points,
        "price_min": price_min,
        "price_max": price_max,
        "price_grid": price_grid,
        "pdf_values": pdf_values,
        "probabilities": probabilities,
        "amplitudes": amplitudes,
    }


if __name__ == "__main__":
    result = discretize_bsm_price_distribution()

    np.set_printoptions(precision=8, suppress=True)

    print("Price grid:")
    print(result["price_grid"])
    print("\nNormalized probabilities:")
    print(result["probabilities"])
    print("\nAmplitudes:")
    print(result["amplitudes"])
    print("\nProbability sum:", np.sum(result["probabilities"]))
    print("Amplitude norm:", np.sum(result["amplitudes"] ** 2))


Price grid:
[439.48215271 457.96346145 476.44477019 494.92607893 513.40738767
 531.88869641 550.37000515 568.85131389]

Normalized probabilities:
[0.0034625  0.03990938 0.1760399  0.33019113 0.28857634 0.12727301
 0.03037597 0.00417177]

Amplitudes:
[0.05884302 0.19977333 0.4195711  0.5746226  0.53719302 0.35675343
 0.17428702 0.06458924]

Probability sum: 1.0
Amplitude norm: 1.0


In [1]:
"""Manual state preparation for 3-qubit SPY price loading.

This script follows the project constraints in agent.md:
- quantum framework: pyqpanda3 only
- gate set for this part: RY and CNOT only
- no macro state-loader or packaged controlled-RY gates
"""

from __future__ import annotations

import numpy as np
import pyqpanda3 as pq


def lognormal_price_pdf(
    prices: np.ndarray,
    s0: float,
    r: float,
    sigma: float,
    maturity: float,
) -> np.ndarray:
    """BSM log-normal density for the terminal asset price S_T."""
    mu = np.log(s0) + (r - 0.5 * sigma**2) * maturity
    std = sigma * np.sqrt(maturity)

    log_term = (np.log(prices) - mu) / std
    normalizer = prices * std * np.sqrt(2.0 * np.pi)
    return np.exp(-0.5 * log_term**2) / normalizer


def discretize_bsm_price_distribution(
    s0: float = 500.0,
    r: float = 0.05,
    sigma: float = 0.15,
    maturity: float = 30.0 / 365.0,
    num_qubits: int = 3,
    truncation_sigma: float = 3.0,
) -> dict[str, np.ndarray | float]:
    """Discretize the continuous BSM price density on a uniform grid."""
    num_points = 2**num_qubits
    sqrt_t = np.sqrt(maturity)

    price_min = s0 * np.exp(-truncation_sigma * sigma * sqrt_t)
    price_max = s0 * np.exp(truncation_sigma * sigma * sqrt_t)
    price_grid = np.linspace(price_min, price_max, num_points)

    pdf_values = lognormal_price_pdf(
        prices=price_grid,
        s0=s0,
        r=r,
        sigma=sigma,
        maturity=maturity,
    )

    grid_spacing = price_grid[1] - price_grid[0]
    unnormalized_probabilities = pdf_values * grid_spacing
    probabilities = unnormalized_probabilities / np.sum(unnormalized_probabilities)
    amplitudes = np.sqrt(probabilities)

    return {
        "price_grid": price_grid,
        "probabilities": probabilities,
        "amplitudes": amplitudes,
    }


def _safe_theta(left_weight: float, right_weight: float) -> float:
    """Return 2*atan2(right, left), with 0 for the degenerate zero-zero case."""
    if np.isclose(left_weight, 0.0) and np.isclose(right_weight, 0.0):
        return 0.0
    return 2.0 * np.arctan2(right_weight, left_weight)


def calculate_ry_angles(amplitudes: np.ndarray) -> np.ndarray:
    """Compute the 7 binary-tree RY angles for a 3-qubit real-amplitude state.

    The amplitude ordering is assumed to be:
    [a000, a001, a010, a011, a100, a101, a110, a111],
    where q[0] is the most significant qubit in the binary label.
    """
    amplitudes = np.asarray(amplitudes, dtype=float)
    if amplitudes.shape != (8,):
        raise ValueError("For 3 qubits, amplitudes must be a length-8 vector.")
    if np.any(amplitudes < -1e-12):
        raise ValueError("This loader assumes nonnegative real amplitudes.")

    norm = np.linalg.norm(amplitudes)
    if np.isclose(norm, 0.0):
        raise ValueError("Amplitude vector must not be the zero vector.")
    amplitudes = amplitudes / norm

    a0, a1, a2, a3, a4, a5, a6, a7 = amplitudes

    theta0 = _safe_theta(
        np.linalg.norm([a0, a1, a2, a3]),
        np.linalg.norm([a4, a5, a6, a7]),
    )

    theta1 = _safe_theta(
        np.linalg.norm([a0, a1]),
        np.linalg.norm([a2, a3]),
    )
    theta2 = _safe_theta(
        np.linalg.norm([a4, a5]),
        np.linalg.norm([a6, a7]),
    )

    theta3 = _safe_theta(a0, a1)
    theta4 = _safe_theta(a2, a3)
    theta5 = _safe_theta(a4, a5)
    theta6 = _safe_theta(a6, a7)

    return np.array([theta0, theta1, theta2, theta3, theta4, theta5, theta6])


def _append_single_control_ry_multiplexor(
    prog: pq.QProg,
    control,
    target,
    angle_if_control_0: float,
    angle_if_control_1: float,
) -> None:
    """Compile a 1-control RY multiplexor into RY + CNOT only."""
    alpha0 = 0.5 * (angle_if_control_0 + angle_if_control_1)
    alpha1 = 0.5 * (angle_if_control_0 - angle_if_control_1)

    prog << pq.RY(target, alpha0)
    prog << pq.CNOT(control, target)
    prog << pq.RY(target, alpha1)
    prog << pq.CNOT(control, target)


def _append_double_control_ry_multiplexor(
    prog: pq.QProg,
    control_msb,
    control_lsb,
    target,
    angles_by_branch: np.ndarray,
) -> None:
    """Compile a 2-control RY multiplexor into RY + CNOT only.

    angles_by_branch uses the control order:
    [angle_00, angle_01, angle_10, angle_11]
    with control_msb control_lsb as the binary branch label.
    """
    theta00, theta01, theta10, theta11 = np.asarray(angles_by_branch, dtype=float)

    beta0 = 0.25 * (theta00 + theta01 + theta10 + theta11)
    beta1 = 0.25 * (theta00 - theta01 + theta10 - theta11)
    beta2 = 0.25 * (theta00 - theta01 - theta10 + theta11)
    beta3 = 0.25 * (theta00 + theta01 - theta10 - theta11)

    prog << pq.RY(target, beta0)
    prog << pq.CNOT(control_lsb, target)
    prog << pq.RY(target, beta1)
    prog << pq.CNOT(control_msb, target)
    prog << pq.RY(target, beta2)
    prog << pq.CNOT(control_lsb, target)
    prog << pq.RY(target, beta3)
    prog << pq.CNOT(control_msb, target)


def build_sp_circuit(angles: np.ndarray, q) -> pq.QProg:
    """Build the 3-qubit state-preparation circuit using only RY and CNOT."""
    if len(q) != 3:
        raise ValueError("This circuit builder expects exactly 3 qubits.")

    angles = np.asarray(angles, dtype=float)
    if angles.shape != (7,):
        raise ValueError("Expected 7 tree angles for the 3-qubit loader.")

    theta0, theta1, theta2, theta3, theta4, theta5, theta6 = angles

    prog = pq.QProg()

    # Root split: q[0] decides between the left half and right half of the tree.
    prog << pq.RY(q[0], theta0)

    # Second level: q[1] is rotated conditioned on q[0].
    _append_single_control_ry_multiplexor(
        prog=prog,
        control=q[0],
        target=q[1],
        angle_if_control_0=theta1,
        angle_if_control_1=theta2,
    )

    # Third level: q[2] is rotated conditioned on the branch q[0]q[1].
    _append_double_control_ry_multiplexor(
        prog=prog,
        control_msb=q[0],
        control_lsb=q[1],
        target=q[2],
        angles_by_branch=np.array([theta3, theta4, theta5, theta6]),
    )

    return prog


def _ordered_probabilities_from_dict(
    prob_dict: dict[str, float],
    num_qubits: int,
    reference: np.ndarray | None = None,
) -> tuple[np.ndarray, str]:
    """Convert prob_run_dict output into a dense probability vector.

    If reference is provided, we also test the reversed-bit ordering and keep
    the one with the smaller max error. This makes the comparison robust to
    simulator bitstring conventions.
    """
    labels = [format(index, f"0{num_qubits}b") for index in range(2**num_qubits)]
    direct = np.array([prob_dict[label] for label in labels], dtype=float)

    if reference is None:
        return direct, "direct"

    reversed_bits = np.array([prob_dict[label[::-1]] for label in labels], dtype=float)

    direct_error = np.max(np.abs(direct - reference))
    reversed_error = np.max(np.abs(reversed_bits - reference))

    if direct_error <= reversed_error:
        return direct, "direct"
    return reversed_bits, "bit-reversed"


if __name__ == "__main__":
    np.set_printoptions(precision=8, suppress=True)

    distribution = discretize_bsm_price_distribution()
    target_probabilities = distribution["probabilities"]
    target_amplitudes = distribution["amplitudes"]

    angles = calculate_ry_angles(target_amplitudes)

    print("Target amplitudes:")
    print(target_amplitudes)
    print("\nTree angles theta_0 ... theta_6:")
    print(angles)

    machine = pq.CPUQVM()
    machine.init_qvm()

    q = machine.qAlloc_many(3)
    prog = build_sp_circuit(angles, q)

    prob_dict = machine.prob_run_dict(prog, q, -1)
    quantum_probabilities, ordering = _ordered_probabilities_from_dict(
        prob_dict=prob_dict,
        num_qubits=3,
        reference=target_probabilities,
    )

    abs_error = np.abs(quantum_probabilities - target_probabilities)

    print("\nRaw prob_run_dict output:")
    print(prob_dict)
    print("\nSelected ordering for comparison:", ordering)
    print("\nTarget probabilities:")
    print(target_probabilities)
    print("\nQuantum probabilities:")
    print(quantum_probabilities)
    print("\nAbsolute error:")
    print(abs_error)
    print("\nMax absolute error:", np.max(abs_error))
    print("L2 error:", np.linalg.norm(abs_error))


Target amplitudes:
[0.05884302 0.19977333 0.4195711  0.5746226  0.53719302 0.35675343
 0.17428702 0.06458924]

Tree angles theta_0 ... theta_6:
[1.47142704 2.5720916  0.5612511  2.56869599 1.88021782 1.17245631
 0.70979967]


AttributeError: module 'pyqpanda3' has no attribute 'CPUQVM'

In [2]:
"""Manual state preparation for 3-qubit SPY price loading.

This script follows the project constraints in agent.md:
- quantum framework: pyqpanda3 only
- gate set for this part: RY and CNOT only
- no macro state-loader or packaged controlled-RY gates
"""

from __future__ import annotations

import numpy as np
import pyqpanda3 as pq

# pyqpanda3 has version differences:
# - old style: symbols such as CPUQVM/QProg/RY/CNOT are exported at top level
# - new style: these symbols live under pyqpanda3.core
QP = pq if all(hasattr(pq, name) for name in ("CPUQVM", "QProg", "RY", "CNOT")) else pq.core


def lognormal_price_pdf(
    prices: np.ndarray,
    s0: float,
    r: float,
    sigma: float,
    maturity: float,
) -> np.ndarray:
    """BSM log-normal density for the terminal asset price S_T."""
    mu = np.log(s0) + (r - 0.5 * sigma**2) * maturity
    std = sigma * np.sqrt(maturity)

    log_term = (np.log(prices) - mu) / std
    normalizer = prices * std * np.sqrt(2.0 * np.pi)
    return np.exp(-0.5 * log_term**2) / normalizer


def discretize_bsm_price_distribution(
    s0: float = 500.0,
    r: float = 0.05,
    sigma: float = 0.15,
    maturity: float = 30.0 / 365.0,
    num_qubits: int = 3,
    truncation_sigma: float = 3.0,
) -> dict[str, np.ndarray | float]:
    """Discretize the continuous BSM price density on a uniform grid."""
    num_points = 2**num_qubits
    sqrt_t = np.sqrt(maturity)

    price_min = s0 * np.exp(-truncation_sigma * sigma * sqrt_t)
    price_max = s0 * np.exp(truncation_sigma * sigma * sqrt_t)
    price_grid = np.linspace(price_min, price_max, num_points)

    pdf_values = lognormal_price_pdf(
        prices=price_grid,
        s0=s0,
        r=r,
        sigma=sigma,
        maturity=maturity,
    )

    grid_spacing = price_grid[1] - price_grid[0]
    unnormalized_probabilities = pdf_values * grid_spacing
    probabilities = unnormalized_probabilities / np.sum(unnormalized_probabilities)
    amplitudes = np.sqrt(probabilities)

    return {
        "price_grid": price_grid,
        "probabilities": probabilities,
        "amplitudes": amplitudes,
    }


def _safe_theta(left_weight: float, right_weight: float) -> float:
    """Return 2*atan2(right, left), with 0 for the degenerate zero-zero case."""
    if np.isclose(left_weight, 0.0) and np.isclose(right_weight, 0.0):
        return 0.0
    return 2.0 * np.arctan2(right_weight, left_weight)


def calculate_ry_angles(amplitudes: np.ndarray) -> np.ndarray:
    """Compute the 7 binary-tree RY angles for a 3-qubit real-amplitude state.

    The amplitude ordering is assumed to be:
    [a000, a001, a010, a011, a100, a101, a110, a111],
    where q[0] is the most significant qubit in the binary label.
    """
    amplitudes = np.asarray(amplitudes, dtype=float)
    if amplitudes.shape != (8,):
        raise ValueError("For 3 qubits, amplitudes must be a length-8 vector.")
    if np.any(amplitudes < -1e-12):
        raise ValueError("This loader assumes nonnegative real amplitudes.")

    norm = np.linalg.norm(amplitudes)
    if np.isclose(norm, 0.0):
        raise ValueError("Amplitude vector must not be the zero vector.")
    amplitudes = amplitudes / norm

    a0, a1, a2, a3, a4, a5, a6, a7 = amplitudes

    theta0 = _safe_theta(
        np.linalg.norm([a0, a1, a2, a3]),
        np.linalg.norm([a4, a5, a6, a7]),
    )

    theta1 = _safe_theta(
        np.linalg.norm([a0, a1]),
        np.linalg.norm([a2, a3]),
    )
    theta2 = _safe_theta(
        np.linalg.norm([a4, a5]),
        np.linalg.norm([a6, a7]),
    )

    theta3 = _safe_theta(a0, a1)
    theta4 = _safe_theta(a2, a3)
    theta5 = _safe_theta(a4, a5)
    theta6 = _safe_theta(a6, a7)

    return np.array([theta0, theta1, theta2, theta3, theta4, theta5, theta6])


def _append_single_control_ry_multiplexor(
    prog,
    control,
    target,
    angle_if_control_0: float,
    angle_if_control_1: float,
) -> None:
    """Compile a 1-control RY multiplexor into RY + CNOT only."""
    alpha0 = 0.5 * (angle_if_control_0 + angle_if_control_1)
    alpha1 = 0.5 * (angle_if_control_0 - angle_if_control_1)

    prog << QP.RY(target, alpha0)
    prog << QP.CNOT(control, target)
    prog << QP.RY(target, alpha1)
    prog << QP.CNOT(control, target)


def _append_double_control_ry_multiplexor(
    prog,
    control_msb,
    control_lsb,
    target,
    angles_by_branch: np.ndarray,
) -> None:
    """Compile a 2-control RY multiplexor into RY + CNOT only.

    angles_by_branch uses the control order:
    [angle_00, angle_01, angle_10, angle_11]
    with control_msb control_lsb as the binary branch label.
    """
    theta00, theta01, theta10, theta11 = np.asarray(angles_by_branch, dtype=float)

    beta0 = 0.25 * (theta00 + theta01 + theta10 + theta11)
    beta1 = 0.25 * (theta00 - theta01 + theta10 - theta11)
    beta2 = 0.25 * (theta00 - theta01 - theta10 + theta11)
    beta3 = 0.25 * (theta00 + theta01 - theta10 - theta11)

    prog << QP.RY(target, beta0)
    prog << QP.CNOT(control_lsb, target)
    prog << QP.RY(target, beta1)
    prog << QP.CNOT(control_msb, target)
    prog << QP.RY(target, beta2)
    prog << QP.CNOT(control_lsb, target)
    prog << QP.RY(target, beta3)
    prog << QP.CNOT(control_msb, target)


def build_sp_circuit(angles: np.ndarray, q):
    """Build the 3-qubit state-preparation circuit using only RY and CNOT."""
    if len(q) != 3:
        raise ValueError("This circuit builder expects exactly 3 qubits.")

    angles = np.asarray(angles, dtype=float)
    if angles.shape != (7,):
        raise ValueError("Expected 7 tree angles for the 3-qubit loader.")

    theta0, theta1, theta2, theta3, theta4, theta5, theta6 = angles

    prog = QP.QProg() if hasattr(QP, "QProg") else pq.core.QProg()

    # Root split: q[0] decides between the left half and right half of the tree.
    prog << QP.RY(q[0], theta0)

    # Second level: q[1] is rotated conditioned on q[0].
    _append_single_control_ry_multiplexor(
        prog=prog,
        control=q[0],
        target=q[1],
        angle_if_control_0=theta1,
        angle_if_control_1=theta2,
    )

    # Third level: q[2] is rotated conditioned on the branch q[0]q[1].
    _append_double_control_ry_multiplexor(
        prog=prog,
        control_msb=q[0],
        control_lsb=q[1],
        target=q[2],
        angles_by_branch=np.array([theta3, theta4, theta5, theta6]),
    )

    return prog


def create_machine_and_qubits(num_qubits: int = 3):
    """Create a CPU simulator and qubit labels with old/new pyqpanda3 compatibility."""
    machine = QP.CPUQVM()

    if hasattr(machine, "init_qvm"):
        machine.init_qvm()

    if hasattr(machine, "qAlloc_many"):
        q = machine.qAlloc_many(num_qubits)
    else:
        q = list(range(num_qubits))

    return machine, q


def run_probabilities(machine, prog, q) -> dict[str, float]:
    """Run the circuit and return a probability dictionary across 3 qubits."""
    if hasattr(machine, "prob_run_dict"):
        return machine.prob_run_dict(prog, q, -1)

    machine.run(prog, 0)
    result = machine.result()

    if hasattr(result, "get_prob_dict"):
        return result.get_prob_dict(q)

    raise AttributeError("Unable to find a supported probability-readout API in pyqpanda3.")


def _ordered_probabilities_from_dict(
    prob_dict: dict[str, float],
    num_qubits: int,
    reference: np.ndarray | None = None,
) -> tuple[np.ndarray, str]:
    """Convert prob_run_dict output into a dense probability vector.

    If reference is provided, we also test the reversed-bit ordering and keep
    the one with the smaller max error. This makes the comparison robust to
    simulator bitstring conventions.
    """
    labels = [format(index, f"0{num_qubits}b") for index in range(2**num_qubits)]
    direct = np.array([prob_dict[label] for label in labels], dtype=float)

    if reference is None:
        return direct, "direct"

    reversed_bits = np.array([prob_dict[label[::-1]] for label in labels], dtype=float)

    direct_error = np.max(np.abs(direct - reference))
    reversed_error = np.max(np.abs(reversed_bits - reference))

    if direct_error <= reversed_error:
        return direct, "direct"
    return reversed_bits, "bit-reversed"


if __name__ == "__main__":
    np.set_printoptions(precision=8, suppress=True)

    distribution = discretize_bsm_price_distribution()
    target_probabilities = distribution["probabilities"]
    target_amplitudes = distribution["amplitudes"]

    angles = calculate_ry_angles(target_amplitudes)

    print("Target amplitudes:")
    print(target_amplitudes)
    print("\nTree angles theta_0 ... theta_6:")
    print(angles)

    machine, q = create_machine_and_qubits(3)
    prog = build_sp_circuit(angles, q)

    prob_dict = run_probabilities(machine, prog, q)
    quantum_probabilities, ordering = _ordered_probabilities_from_dict(
        prob_dict=prob_dict,
        num_qubits=3,
        reference=target_probabilities,
    )

    abs_error = np.abs(quantum_probabilities - target_probabilities)

    print("\nRaw prob_run_dict output:")
    print(prob_dict)
    print("\nSelected ordering for comparison:", ordering)
    print("\nTarget probabilities:")
    print(target_probabilities)
    print("\nQuantum probabilities:")
    print(quantum_probabilities)
    print("\nAbsolute error:")
    print(abs_error)
    print("\nMax absolute error:", np.max(abs_error))
    print("L2 error:", np.linalg.norm(abs_error))


Target amplitudes:
[0.05884302 0.19977333 0.4195711  0.5746226  0.53719302 0.35675343
 0.17428702 0.06458924]

Tree angles theta_0 ... theta_6:
[1.47142704 2.5720916  0.5612511  2.56869599 1.88021782 1.17245631
 0.70979967]

Raw prob_run_dict output:
{'000': 0.0034625007358196924, '001': 0.2885763394193458, '010': 0.17603990456887741, '011': 0.03037596508970815, '100': 0.0399093815926149, '101': 0.12727300907888908, '110': 0.33019112999015454, '111': 0.004171769524590569}

Selected ordering for comparison: bit-reversed

Target probabilities:
[0.0034625  0.03990938 0.1760399  0.33019113 0.28857634 0.12727301
 0.03037597 0.00417177]

Quantum probabilities:
[0.0034625  0.03990938 0.1760399  0.33019113 0.28857634 0.12727301
 0.03037597 0.00417177]

Absolute error:
[0. 0. 0. 0. 0. 0. 0. 0.]

Max absolute error: 1.1102230246251565e-16
L2 error: 1.3637246461285863e-16


In [3]:
"""State preparation plus payoff encoding for a 3-qubit SPY call option model.

This script is fully manual:
- only pyqpanda3 is used for quantum circuits
- only RY and CNOT are used to build the circuits
- no packaged controlled rotations or state-loader macros are used
"""

from __future__ import annotations

import numpy as np
import pyqpanda3 as pq

QP = pq if all(hasattr(pq, name) for name in ("CPUQVM", "QProg", "RY", "CNOT")) else pq.core


def lognormal_price_pdf(
    prices: np.ndarray,
    s0: float,
    r: float,
    sigma: float,
    maturity: float,
) -> np.ndarray:
    """BSM log-normal density for terminal price S_T."""
    mu = np.log(s0) + (r - 0.5 * sigma**2) * maturity
    std = sigma * np.sqrt(maturity)

    log_term = (np.log(prices) - mu) / std
    normalizer = prices * std * np.sqrt(2.0 * np.pi)
    return np.exp(-0.5 * log_term**2) / normalizer


def discretize_bsm_price_distribution(
    s0: float = 500.0,
    r: float = 0.05,
    sigma: float = 0.15,
    maturity: float = 30.0 / 365.0,
    num_qubits: int = 3,
    truncation_sigma: float = 3.0,
) -> dict[str, np.ndarray | float]:
    """Discretize the BSM price density on a uniform price grid."""
    num_points = 2**num_qubits
    sqrt_t = np.sqrt(maturity)

    price_min = s0 * np.exp(-truncation_sigma * sigma * sqrt_t)
    price_max = s0 * np.exp(truncation_sigma * sigma * sqrt_t)
    price_grid = np.linspace(price_min, price_max, num_points)

    pdf_values = lognormal_price_pdf(
        prices=price_grid,
        s0=s0,
        r=r,
        sigma=sigma,
        maturity=maturity,
    )

    grid_spacing = price_grid[1] - price_grid[0]
    unnormalized_probabilities = pdf_values * grid_spacing
    probabilities = unnormalized_probabilities / np.sum(unnormalized_probabilities)
    amplitudes = np.sqrt(probabilities)

    return {
        "price_grid": price_grid,
        "probabilities": probabilities,
        "amplitudes": amplitudes,
    }


def _safe_theta(left_weight: float, right_weight: float) -> float:
    """Return 2*atan2(right, left), with 0 for the degenerate zero-zero case."""
    if np.isclose(left_weight, 0.0) and np.isclose(right_weight, 0.0):
        return 0.0
    return 2.0 * np.arctan2(right_weight, left_weight)


def calculate_ry_angles(amplitudes: np.ndarray) -> np.ndarray:
    """Compute the 7 binary-tree RY angles for 3-qubit real state preparation."""
    amplitudes = np.asarray(amplitudes, dtype=float)
    if amplitudes.shape != (8,):
        raise ValueError("For 3 qubits, amplitudes must be a length-8 vector.")
    if np.any(amplitudes < -1e-12):
        raise ValueError("This loader assumes nonnegative real amplitudes.")

    norm = np.linalg.norm(amplitudes)
    if np.isclose(norm, 0.0):
        raise ValueError("Amplitude vector must not be the zero vector.")
    amplitudes = amplitudes / norm

    a0, a1, a2, a3, a4, a5, a6, a7 = amplitudes

    theta0 = _safe_theta(
        np.linalg.norm([a0, a1, a2, a3]),
        np.linalg.norm([a4, a5, a6, a7]),
    )
    theta1 = _safe_theta(np.linalg.norm([a0, a1]), np.linalg.norm([a2, a3]))
    theta2 = _safe_theta(np.linalg.norm([a4, a5]), np.linalg.norm([a6, a7]))
    theta3 = _safe_theta(a0, a1)
    theta4 = _safe_theta(a2, a3)
    theta5 = _safe_theta(a4, a5)
    theta6 = _safe_theta(a6, a7)

    return np.array([theta0, theta1, theta2, theta3, theta4, theta5, theta6])


def _append_single_control_ry_multiplexor(
    prog,
    control,
    target,
    angle_if_control_0: float,
    angle_if_control_1: float,
) -> None:
    """Compile a 1-control RY multiplexor into RY + CNOT only."""
    alpha0 = 0.5 * (angle_if_control_0 + angle_if_control_1)
    alpha1 = 0.5 * (angle_if_control_0 - angle_if_control_1)

    prog << QP.RY(target, alpha0)
    prog << QP.CNOT(control, target)
    prog << QP.RY(target, alpha1)
    prog << QP.CNOT(control, target)


def _append_double_control_ry_multiplexor(
    prog,
    control_msb,
    control_lsb,
    target,
    angles_by_branch: np.ndarray,
) -> None:
    """Compile a 2-control RY multiplexor into RY + CNOT only."""
    theta00, theta01, theta10, theta11 = np.asarray(angles_by_branch, dtype=float)

    beta0 = 0.25 * (theta00 + theta01 + theta10 + theta11)
    beta1 = 0.25 * (theta00 - theta01 + theta10 - theta11)
    beta2 = 0.25 * (theta00 - theta01 - theta10 + theta11)
    beta3 = 0.25 * (theta00 + theta01 - theta10 - theta11)

    prog << QP.RY(target, beta0)
    prog << QP.CNOT(control_lsb, target)
    prog << QP.RY(target, beta1)
    prog << QP.CNOT(control_msb, target)
    prog << QP.RY(target, beta2)
    prog << QP.CNOT(control_lsb, target)
    prog << QP.RY(target, beta3)
    prog << QP.CNOT(control_msb, target)


def build_sp_circuit(angles: np.ndarray, q):
    """Build the 3-qubit price-loading circuit using only RY and CNOT."""
    if len(q) != 3:
        raise ValueError("State preparation expects exactly 3 qubits.")

    theta0, theta1, theta2, theta3, theta4, theta5, theta6 = np.asarray(angles, dtype=float)

    prog = QP.QProg()
    prog << QP.RY(q[0], theta0)

    _append_single_control_ry_multiplexor(
        prog=prog,
        control=q[0],
        target=q[1],
        angle_if_control_0=theta1,
        angle_if_control_1=theta2,
    )

    _append_double_control_ry_multiplexor(
        prog=prog,
        control_msb=q[0],
        control_lsb=q[1],
        target=q[2],
        angles_by_branch=np.array([theta3, theta4, theta5, theta6]),
    )

    return prog


def calculate_call_payoffs(price_grid: np.ndarray, strike: float = 500.0) -> np.ndarray:
    """European call payoff max(S-K, 0) on the discrete price grid."""
    return np.maximum(np.asarray(price_grid, dtype=float) - strike, 0.0)


def calculate_gamma_angles(
    payoff_list: np.ndarray,
    scaling_constant: float = 0.5,
) -> tuple[np.ndarray, float, np.ndarray]:
    """Convert payoffs into branch rotation angles gamma_i.

    gamma_i is defined by
    sin^2(gamma_i / 2) = c * payoff_i / max_payoff.
    """
    payoff_list = np.asarray(payoff_list, dtype=float)
    if payoff_list.ndim != 1:
        raise ValueError("payoff_list must be a one-dimensional array.")
    if scaling_constant <= 0.0 or scaling_constant > 1.0:
        raise ValueError("scaling_constant must lie in (0, 1].")

    max_payoff = float(np.max(payoff_list))
    if np.isclose(max_payoff, 0.0):
        scaled_payoffs = np.zeros_like(payoff_list)
        gamma_list = np.zeros_like(payoff_list)
    else:
        scaled_payoffs = scaling_constant * payoff_list / max_payoff
        if np.any(scaled_payoffs > 1.0 + 1e-12):
            raise ValueError("Scaled payoff exceeds 1, decrease the scaling constant.")
        scaled_payoffs = np.clip(scaled_payoffs, 0.0, 1.0)
        gamma_list = 2.0 * np.arcsin(np.sqrt(scaled_payoffs))

    return gamma_list, max_payoff, scaled_payoffs


def gray_code_sequence(num_controls: int) -> list[int]:
    """Return the standard binary-reflected Gray code sequence."""
    return [index ^ (index >> 1) for index in range(2**num_controls)]


def _gray_ordered_walsh_hadamard_matrix(num_controls: int) -> np.ndarray:
    """Return the Gray-ordered Walsh-Hadamard transform matrix.

    For a k-control uniformly controlled RY rotation, the base angles satisfy

        delta = M * gamma,

    where

        M[r, c] = 2^{-k} (-1)^{popcount(gray_r & c)}.

    Row r is indexed by the r-th Gray code word, while column c is indexed by
    the standard binary branch label.
    """
    num_branches = 2**num_controls
    matrix = np.empty((num_branches, num_branches), dtype=float)
    gray_codes = gray_code_sequence(num_controls)

    for row, gray_word in enumerate(gray_codes):
        for col in range(num_branches):
            parity = (gray_word & col).bit_count() % 2
            matrix[row, col] = 1.0 if parity == 0 else -1.0

    return matrix / num_branches


def calculate_payoff_angles(gamma_list: np.ndarray) -> np.ndarray:
    """Convert branch angles gamma_i into Gray-code multiplexor angles delta_i."""
    gamma_list = np.asarray(gamma_list, dtype=float)
    if gamma_list.ndim != 1:
        raise ValueError("gamma_list must be one-dimensional.")

    num_branches = gamma_list.size
    num_controls = int(np.log2(num_branches))
    if 2**num_controls != num_branches:
        raise ValueError("gamma_list length must be a power of two.")

    transform = _gray_ordered_walsh_hadamard_matrix(num_controls)
    return transform @ gamma_list


def _control_for_gray_transition(
    q_control_list,
    current_gray_word: int,
    next_gray_word: int,
):
    """Map a Gray-code bit flip to the corresponding control qubit.

    q_control_list is ordered as [MSB, ..., LSB].
    """
    toggle_mask = current_gray_word ^ next_gray_word
    if toggle_mask == 0 or toggle_mask & (toggle_mask - 1):
        raise ValueError("Gray-code transitions must flip exactly one bit.")

    toggled_bit_from_lsb = toggle_mask.bit_length() - 1
    return q_control_list[-1 - toggled_bit_from_lsb]


def build_payoff_circuit(delta_angles: np.ndarray, q_control_list, q_target):
    """Build a 3-control uniformly controlled RY payoff circuit.

    The circuit alternates RY and CNOT gates in Gray-code order:
    RY(delta_0) -> CNOT -> RY(delta_1) -> CNOT -> ... -> RY(delta_7) -> CNOT.
    """
    delta_angles = np.asarray(delta_angles, dtype=float)
    num_controls = len(q_control_list)
    if delta_angles.size != 2**num_controls:
        raise ValueError("delta_angles length must be 2**len(q_control_list).")

    gray_codes = gray_code_sequence(num_controls)
    prog = QP.QProg()

    for index, angle in enumerate(delta_angles):
        prog << QP.RY(q_target, float(angle))

        current_gray_word = gray_codes[index]
        next_gray_word = gray_codes[(index + 1) % len(gray_codes)]
        control_qubit = _control_for_gray_transition(
            q_control_list=q_control_list,
            current_gray_word=current_gray_word,
            next_gray_word=next_gray_word,
        )
        prog << QP.CNOT(control_qubit, q_target)

    return prog


def create_machine_and_qubits(num_qubits: int):
    """Create a CPUQVM and a qubit container compatible with old/new pyqpanda3 APIs."""
    machine = QP.CPUQVM()
    if hasattr(machine, "init_qvm"):
        machine.init_qvm()

    if hasattr(machine, "qAlloc_many"):
        q = machine.qAlloc_many(num_qubits)
    else:
        q = list(range(num_qubits))

    return machine, q


def run_probabilities(machine, prog, q) -> dict[str, float]:
    """Run a quantum program and return the full probability dictionary."""
    if hasattr(machine, "prob_run_dict"):
        return machine.prob_run_dict(prog, q, -1)

    machine.run(prog, 0)
    result = machine.result()
    return result.get_prob_dict(q)


def build_expected_full_distribution(
    price_probabilities: np.ndarray,
    scaled_payoffs: np.ndarray,
) -> dict[str, float]:
    """Return the expected 4-qubit probability distribution.

    The qubit list is assumed to be [q0, q1, q2, q3], while pyqpanda3 prints
    bitstrings as q3 q2 q1 q0 when get_prob_dict([0,1,2,3]) is used.
    """
    expected = {}
    for branch_index, (branch_prob, scaled_payoff) in enumerate(zip(price_probabilities, scaled_payoffs)):
        branch_bits = format(branch_index, "03b")  # q0 q1 q2 in the math convention
        simulator_control_bits = branch_bits[::-1]  # q2 q1 q0 in pyqpanda3 output strings

        expected["0" + simulator_control_bits] = branch_prob * (1.0 - scaled_payoff)
        expected["1" + simulator_control_bits] = branch_prob * scaled_payoff

    return expected


def total_probability_target_one(prob_dict: dict[str, float]) -> float:
    """Extract the total probability that q[3] = 1.

    With get_prob_dict([0,1,2,3]), pyqpanda3 prints bitstrings as q3 q2 q1 q0,
    so q[3] is the leading bit.
    """
    return sum(probability for bitstring, probability in prob_dict.items() if bitstring[0] == "1")


if __name__ == "__main__":
    np.set_printoptions(precision=8, suppress=True)

    strike = 500.0
    scaling_constant = 0.5

    distribution = discretize_bsm_price_distribution()
    price_grid = distribution["price_grid"]
    price_probabilities = distribution["probabilities"]
    price_amplitudes = distribution["amplitudes"]

    sp_angles = calculate_ry_angles(price_amplitudes)

    payoff_list = calculate_call_payoffs(price_grid, strike=strike)
    gamma_list, max_payoff, scaled_payoffs = calculate_gamma_angles(
        payoff_list,
        scaling_constant=scaling_constant,
    )
    delta_angles = calculate_payoff_angles(gamma_list)

    theoretical_target_one_probability = float(np.sum(price_probabilities * scaled_payoffs))

    machine, q = create_machine_and_qubits(4)

    prog = QP.QProg()
    prog << build_sp_circuit(sp_angles, q[:3])
    prog << build_payoff_circuit(delta_angles, q[:3], q[3])

    prob_dict = run_probabilities(machine, prog, q)
    expected_prob_dict = build_expected_full_distribution(price_probabilities, scaled_payoffs)

    raw_labels = [format(index, "04b") for index in range(16)]
    quantum_probabilities = np.array([prob_dict[label] for label in raw_labels], dtype=float)
    expected_probabilities = np.array([expected_prob_dict[label] for label in raw_labels], dtype=float)
    full_abs_error = np.abs(quantum_probabilities - expected_probabilities)

    quantum_target_one_probability = total_probability_target_one(prob_dict)
    target_one_abs_error = abs(quantum_target_one_probability - theoretical_target_one_probability)

    print("Price grid:")
    print(price_grid)
    print("\nPrice probabilities:")
    print(price_probabilities)
    print("\nCall payoff list:")
    print(payoff_list)
    print("\nGamma angles:")
    print(gamma_list)
    print("\nDelta angles:")
    print(delta_angles)
    print("\nRaw 4-qubit probability distribution:")
    print(prob_dict)
    print("\nExpected 4-qubit probability distribution:")
    print(expected_prob_dict)
    print("\nFull-distribution absolute error:")
    print(full_abs_error)
    print("\nMax full-distribution absolute error:", np.max(full_abs_error))
    print("\nTheoretical P(q[3]=1):", theoretical_target_one_probability)
    print("Quantum P(q[3]=1):", quantum_target_one_probability)
    print("Absolute error on P(q[3]=1):", target_one_abs_error)


Price grid:
[439.48215271 457.96346145 476.44477019 494.92607893 513.40738767
 531.88869641 550.37000515 568.85131389]

Price probabilities:
[0.0034625  0.03990938 0.1760399  0.33019113 0.28857634 0.12727301
 0.03037597 0.00417177]

Call payoff list:
[ 0.          0.          0.          0.         13.40738767 31.88869641
 50.37000515 68.85131389]

Gamma angles:
[0.         0.         0.         0.         0.63466491 1.00410095
 1.29904026 1.57079633]

Delta angles:
[ 0.56357531 -0.08014901 -0.01221    -0.15388384  0.15388384  0.01221
  0.08014901 -0.56357531]

Raw 4-qubit probability distribution:
{'0000': 0.0034625007358196924, '0001': 0.2604791642432872, '0010': 0.1760399045688772, '0011': 0.019264793555293775, '0100': 0.03990938159261489, '0101': 0.09779956755405206, '0110': 0.33019112999015443, '0111': 0.002085884762295283, '1000': 0.0, '1001': 0.028097175176058736, '1010': 1.7333369499485123e-33, '1011': 0.011111171534414365, '1100': 1.2037062152420224e-33, '1101': 0.029473441524

In [4]:
"""Core Grover operator for IQAE-based SPY option pricing.

This script constructs the oracle-ready Grover iterate

    Q = - A S_0 A^\dagger S_f

using only low-level single-qubit gates together with CNOT.
No state-loader macro, controlled-rotation macro, or multi-controlled gate macro
is used anywhere in the implementation.
"""

from __future__ import annotations

import itertools
import math
import numpy as np
import pyqpanda3 as pq

QP = pq if all(hasattr(pq, name) for name in ("CPUQVM", "QProg", "RY", "CNOT")) else pq.core


def lognormal_price_pdf(
    prices: np.ndarray,
    s0: float,
    r: float,
    sigma: float,
    maturity: float,
) -> np.ndarray:
    """BSM log-normal density for terminal price S_T."""
    mu = np.log(s0) + (r - 0.5 * sigma**2) * maturity
    std = sigma * np.sqrt(maturity)

    log_term = (np.log(prices) - mu) / std
    normalizer = prices * std * np.sqrt(2.0 * np.pi)
    return np.exp(-0.5 * log_term**2) / normalizer


def discretize_bsm_price_distribution(
    s0: float = 500.0,
    r: float = 0.05,
    sigma: float = 0.15,
    maturity: float = 30.0 / 365.0,
    num_qubits: int = 3,
    truncation_sigma: float = 3.0,
) -> dict[str, np.ndarray | float]:
    """Discretize the BSM price density on a uniform price grid."""
    num_points = 2**num_qubits
    sqrt_t = np.sqrt(maturity)

    price_min = s0 * np.exp(-truncation_sigma * sigma * sqrt_t)
    price_max = s0 * np.exp(truncation_sigma * sigma * sqrt_t)
    price_grid = np.linspace(price_min, price_max, num_points)

    pdf_values = lognormal_price_pdf(
        prices=price_grid,
        s0=s0,
        r=r,
        sigma=sigma,
        maturity=maturity,
    )

    grid_spacing = price_grid[1] - price_grid[0]
    unnormalized_probabilities = pdf_values * grid_spacing
    probabilities = unnormalized_probabilities / np.sum(unnormalized_probabilities)
    amplitudes = np.sqrt(probabilities)

    return {
        "price_grid": price_grid,
        "probabilities": probabilities,
        "amplitudes": amplitudes,
    }


def _safe_theta(left_weight: float, right_weight: float) -> float:
    """Return 2*atan2(right, left), with 0 for the degenerate zero-zero case."""
    if np.isclose(left_weight, 0.0) and np.isclose(right_weight, 0.0):
        return 0.0
    return 2.0 * np.arctan2(right_weight, left_weight)


def calculate_ry_angles(amplitudes: np.ndarray) -> np.ndarray:
    """Compute the 7 binary-tree RY angles for the 3-qubit price state."""
    amplitudes = np.asarray(amplitudes, dtype=float)
    amplitudes = amplitudes / np.linalg.norm(amplitudes)

    a0, a1, a2, a3, a4, a5, a6, a7 = amplitudes

    theta0 = _safe_theta(np.linalg.norm([a0, a1, a2, a3]), np.linalg.norm([a4, a5, a6, a7]))
    theta1 = _safe_theta(np.linalg.norm([a0, a1]), np.linalg.norm([a2, a3]))
    theta2 = _safe_theta(np.linalg.norm([a4, a5]), np.linalg.norm([a6, a7]))
    theta3 = _safe_theta(a0, a1)
    theta4 = _safe_theta(a2, a3)
    theta5 = _safe_theta(a4, a5)
    theta6 = _safe_theta(a6, a7)

    return np.array([theta0, theta1, theta2, theta3, theta4, theta5, theta6])


def _append_single_control_ry_multiplexor(
    prog,
    control,
    target,
    angle_if_control_0: float,
    angle_if_control_1: float,
) -> None:
    """Compile a 1-control RY multiplexor into RY + CNOT only."""
    alpha0 = 0.5 * (angle_if_control_0 + angle_if_control_1)
    alpha1 = 0.5 * (angle_if_control_0 - angle_if_control_1)

    prog << QP.RY(target, alpha0)
    prog << QP.CNOT(control, target)
    prog << QP.RY(target, alpha1)
    prog << QP.CNOT(control, target)


def _append_inverse_single_control_ry_multiplexor(
    prog,
    control,
    target,
    angle_if_control_0: float,
    angle_if_control_1: float,
) -> None:
    """Append the inverse of the 1-control RY multiplexor."""
    alpha0 = 0.5 * (angle_if_control_0 + angle_if_control_1)
    alpha1 = 0.5 * (angle_if_control_0 - angle_if_control_1)

    prog << QP.CNOT(control, target)
    prog << QP.RY(target, -alpha1)
    prog << QP.CNOT(control, target)
    prog << QP.RY(target, -alpha0)


def _append_double_control_ry_multiplexor(
    prog,
    control_msb,
    control_lsb,
    target,
    angles_by_branch: np.ndarray,
) -> None:
    """Compile a 2-control RY multiplexor into RY + CNOT only."""
    theta00, theta01, theta10, theta11 = np.asarray(angles_by_branch, dtype=float)

    beta0 = 0.25 * (theta00 + theta01 + theta10 + theta11)
    beta1 = 0.25 * (theta00 - theta01 + theta10 - theta11)
    beta2 = 0.25 * (theta00 - theta01 - theta10 + theta11)
    beta3 = 0.25 * (theta00 + theta01 - theta10 - theta11)

    prog << QP.RY(target, beta0)
    prog << QP.CNOT(control_lsb, target)
    prog << QP.RY(target, beta1)
    prog << QP.CNOT(control_msb, target)
    prog << QP.RY(target, beta2)
    prog << QP.CNOT(control_lsb, target)
    prog << QP.RY(target, beta3)
    prog << QP.CNOT(control_msb, target)


def _append_inverse_double_control_ry_multiplexor(
    prog,
    control_msb,
    control_lsb,
    target,
    angles_by_branch: np.ndarray,
) -> None:
    """Append the inverse of the 2-control RY multiplexor."""
    theta00, theta01, theta10, theta11 = np.asarray(angles_by_branch, dtype=float)

    beta0 = 0.25 * (theta00 + theta01 + theta10 + theta11)
    beta1 = 0.25 * (theta00 - theta01 + theta10 - theta11)
    beta2 = 0.25 * (theta00 - theta01 - theta10 + theta11)
    beta3 = 0.25 * (theta00 + theta01 - theta10 - theta11)

    prog << QP.CNOT(control_msb, target)
    prog << QP.RY(target, -beta3)
    prog << QP.CNOT(control_lsb, target)
    prog << QP.RY(target, -beta2)
    prog << QP.CNOT(control_msb, target)
    prog << QP.RY(target, -beta1)
    prog << QP.CNOT(control_lsb, target)
    prog << QP.RY(target, -beta0)


def build_sp_circuit(angles: np.ndarray, q):
    """Build the 3-qubit state-preparation circuit."""
    theta0, theta1, theta2, theta3, theta4, theta5, theta6 = np.asarray(angles, dtype=float)

    prog = QP.QProg()
    prog << QP.RY(q[0], theta0)
    _append_single_control_ry_multiplexor(prog, q[0], q[1], theta1, theta2)
    _append_double_control_ry_multiplexor(prog, q[0], q[1], q[2], np.array([theta3, theta4, theta5, theta6]))
    return prog


def build_inverse_sp_circuit(angles: np.ndarray, q):
    """Build the inverse of the 3-qubit state-preparation circuit."""
    theta0, theta1, theta2, theta3, theta4, theta5, theta6 = np.asarray(angles, dtype=float)

    prog = QP.QProg()
    _append_inverse_double_control_ry_multiplexor(
        prog,
        q[0],
        q[1],
        q[2],
        np.array([theta3, theta4, theta5, theta6]),
    )
    _append_inverse_single_control_ry_multiplexor(prog, q[0], q[1], theta1, theta2)
    prog << QP.RY(q[0], -theta0)
    return prog


def calculate_call_payoffs(price_grid: np.ndarray, strike: float = 500.0) -> np.ndarray:
    """European call payoff max(S-K, 0)."""
    return np.maximum(np.asarray(price_grid, dtype=float) - strike, 0.0)


def calculate_gamma_angles(
    payoff_list: np.ndarray,
    scaling_constant: float = 0.5,
) -> tuple[np.ndarray, float, np.ndarray]:
    """Map call payoffs to target-qubit branch angles gamma_i."""
    payoff_list = np.asarray(payoff_list, dtype=float)
    max_payoff = float(np.max(payoff_list))

    if np.isclose(max_payoff, 0.0):
        scaled_payoffs = np.zeros_like(payoff_list)
        gamma_list = np.zeros_like(payoff_list)
    else:
        scaled_payoffs = scaling_constant * payoff_list / max_payoff
        scaled_payoffs = np.clip(scaled_payoffs, 0.0, 1.0)
        gamma_list = 2.0 * np.arcsin(np.sqrt(scaled_payoffs))

    return gamma_list, max_payoff, scaled_payoffs


def gray_code_sequence(num_controls: int) -> list[int]:
    """Return the standard binary-reflected Gray code sequence."""
    return [index ^ (index >> 1) for index in range(2**num_controls)]


def _gray_ordered_walsh_hadamard_matrix(num_controls: int) -> np.ndarray:
    """Gray-ordered Walsh-Hadamard transform for uniformly controlled rotations."""
    num_branches = 2**num_controls
    matrix = np.empty((num_branches, num_branches), dtype=float)
    gray_codes = gray_code_sequence(num_controls)

    for row, gray_word in enumerate(gray_codes):
        for col in range(num_branches):
            parity = (gray_word & col).bit_count() % 2
            matrix[row, col] = 1.0 if parity == 0 else -1.0

    return matrix / num_branches


def calculate_payoff_angles(gamma_list: np.ndarray) -> np.ndarray:
    """Compute the delta angles for the 3-control uniformly controlled RY."""
    gamma_list = np.asarray(gamma_list, dtype=float)
    num_controls = int(np.log2(gamma_list.size))
    transform = _gray_ordered_walsh_hadamard_matrix(num_controls)
    return transform @ gamma_list


def _control_sequence_for_uniform_rotation(q_control_list) -> list[int]:
    """Return the Gray-code control-toggle sequence for uniformly controlled RY."""
    num_controls = len(q_control_list)
    gray_codes = gray_code_sequence(num_controls)
    sequence = []

    for index in range(len(gray_codes)):
        current_gray_word = gray_codes[index]
        next_gray_word = gray_codes[(index + 1) % len(gray_codes)]
        toggle_mask = current_gray_word ^ next_gray_word
        toggled_bit_from_lsb = toggle_mask.bit_length() - 1
        sequence.append(q_control_list[-1 - toggled_bit_from_lsb])

    return sequence


def build_payoff_circuit(delta_angles: np.ndarray, q_control_list, q_target):
    """Build the 3-control payoff oracle using only RY and CNOT."""
    delta_angles = np.asarray(delta_angles, dtype=float)
    control_sequence = _control_sequence_for_uniform_rotation(q_control_list)

    prog = QP.QProg()
    for angle, control in zip(delta_angles, control_sequence):
        prog << QP.RY(q_target, float(angle))
        prog << QP.CNOT(control, q_target)

    return prog


def build_inverse_payoff_circuit(delta_angles: np.ndarray, q_control_list, q_target):
    """Build the inverse of the payoff multiplexor."""
    delta_angles = np.asarray(delta_angles, dtype=float)
    control_sequence = _control_sequence_for_uniform_rotation(q_control_list)

    prog = QP.QProg()
    for angle, control in reversed(list(zip(delta_angles, control_sequence))):
        prog << QP.CNOT(control, q_target)
        prog << QP.RY(q_target, float(-angle))

    return prog


def build_A_circuit(sp_angles: np.ndarray, delta_angles: np.ndarray, q_control_list, q_target):
    """Build the full state-loading operator A = SP + Payoff."""
    prog = QP.QProg()
    prog << build_sp_circuit(sp_angles, q_control_list)
    prog << build_payoff_circuit(delta_angles, q_control_list, q_target)
    return prog


def build_inverse_A_circuit(
    sp_angles: np.ndarray,
    delta_angles: np.ndarray,
    q_control_list,
    q_target,
):
    """Build the exact inverse A^\u2020 by reversing all low-level gates manually."""
    prog = QP.QProg()
    prog << build_inverse_payoff_circuit(delta_angles, q_control_list, q_target)
    prog << build_inverse_sp_circuit(sp_angles, q_control_list)
    return prog


def _append_phase_product(prog, qubit_subset, alpha: float) -> None:
    """Append exp(i alpha Z_{subset}) using only CNOT and RZ."""
    qubit_subset = list(qubit_subset)
    target = qubit_subset[-1]

    for control in qubit_subset[:-1]:
        prog << QP.CNOT(control, target)
    prog << QP.RZ(target, -2.0 * alpha)
    for control in reversed(qubit_subset[:-1]):
        prog << QP.CNOT(control, target)


def build_multi_controlled_z_all_ones(qubits):
    """Build a phase flip on |11...1> using phase-polynomial synthesis.

    Up to a physically irrelevant global phase, the operator is the exact
    multi-controlled Z reflection on the all-ones basis state.
    """
    qubits = list(qubits)
    num_qubits = len(qubits)
    prog = QP.QProg()

    for subset_size in range(1, num_qubits + 1):
        for subset_indices in itertools.combinations(range(num_qubits), subset_size):
            subset_qubits = [qubits[index] for index in subset_indices]
            alpha = math.pi / (2**num_qubits) * ((-1) ** subset_size)
            _append_phase_product(prog, subset_qubits, alpha)

    return prog


def build_S0_circuit(qubits):
    """Build the all-zero reflection S_0 using X wrappers and an exact C^3Z."""
    qubits = list(qubits)
    prog = QP.QProg()

    for qubit in qubits:
        prog << QP.X(qubit)

    prog << build_multi_controlled_z_all_ones(qubits)

    for qubit in qubits:
        prog << QP.X(qubit)

    return prog


def build_Sf_circuit(q_target):
    """Build S_f, the phase flip on the good subspace q_target = 1."""
    prog = QP.QProg()
    prog << QP.Z(q_target)
    return prog


def build_grover_Q(
    sp_angles: np.ndarray,
    delta_angles: np.ndarray,
    q_control_list,
    q_target,
):
    """Build the Grover iterate Q = -A S_0 A^\u2020 S_f.

    The leading global phase -1 is omitted, because it has no physical effect.
    """
    all_qubits = list(q_control_list) + [q_target]

    prog = QP.QProg()
    prog << build_Sf_circuit(q_target)
    prog << build_inverse_A_circuit(sp_angles, delta_angles, q_control_list, q_target)
    prog << build_S0_circuit(all_qubits)
    prog << build_A_circuit(sp_angles, delta_angles, q_control_list, q_target)
    return prog


def create_machine_and_qubits(num_qubits: int):
    """Create a CPUQVM with old/new pyqpanda3 compatibility."""
    machine = QP.CPUQVM()
    if hasattr(machine, "init_qvm"):
        machine.init_qvm()

    if hasattr(machine, "qAlloc_many"):
        q = machine.qAlloc_many(num_qubits)
    else:
        q = list(range(num_qubits))

    return machine, q


def run_probabilities(machine, prog, q) -> dict[str, float]:
    """Run a quantum program and return its full probability dictionary."""
    if hasattr(machine, "prob_run_dict"):
        return machine.prob_run_dict(prog, q, -1)

    machine.run(prog, 0)
    return machine.result().get_prob_dict(q)


def total_probability_target_one(prob_dict: dict[str, float]) -> float:
    """Extract the total probability that q[3] = 1.

    In pyqpanda3's get_prob_dict([0,1,2,3]), bitstrings are printed as q3 q2 q1 q0.
    Hence q[3] corresponds to the leading bit.
    """
    return sum(probability for bitstring, probability in prob_dict.items() if bitstring[0] == "1")


if __name__ == "__main__":
    np.set_printoptions(precision=8, suppress=True)

    strike = 500.0
    scaling_constant = 0.5

    distribution = discretize_bsm_price_distribution()
    price_grid = distribution["price_grid"]
    price_probabilities = distribution["probabilities"]
    price_amplitudes = distribution["amplitudes"]

    sp_angles = calculate_ry_angles(price_amplitudes)

    payoff_list = calculate_call_payoffs(price_grid, strike=strike)
    gamma_list, max_payoff, scaled_payoffs = calculate_gamma_angles(
        payoff_list,
        scaling_constant=scaling_constant,
    )
    delta_angles = calculate_payoff_angles(gamma_list)

    initial_success_probability = float(np.sum(price_probabilities * scaled_payoffs))
    theta = math.asin(math.sqrt(initial_success_probability))
    one_grover_theory = math.sin(3.0 * theta) ** 2

    machine, q = create_machine_and_qubits(4)

    prog_A = QP.QProg()
    prog_A << build_A_circuit(sp_angles, delta_angles, q[:3], q[3])
    prob_dict_A = run_probabilities(machine, prog_A, q)
    success_probability_A = total_probability_target_one(prob_dict_A)

    machine_after_Q, q_after_Q = create_machine_and_qubits(4)
    prog_AQ = QP.QProg()
    prog_AQ << build_A_circuit(sp_angles, delta_angles, q_after_Q[:3], q_after_Q[3])
    prog_AQ << build_grover_Q(sp_angles, delta_angles, q_after_Q[:3], q_after_Q[3])
    prob_dict_AQ = run_probabilities(machine_after_Q, prog_AQ, q_after_Q)
    success_probability_AQ = total_probability_target_one(prob_dict_AQ)

    print("Price grid:")
    print(price_grid)
    print("\nPayoff list:")
    print(payoff_list)
    print("\nGamma angles:")
    print(gamma_list)
    print("\nDelta angles:")
    print(delta_angles)
    print("\nInitial success probability P(q[3]=1) after A:")
    print(success_probability_A)
    print("Classical theory for A:")
    print(initial_success_probability)
    print("\nSuccess probability P(q[3]=1) after A followed by one Q:")
    print(success_probability_AQ)
    print("Amplitude-amplification theory sin^2(3 theta):")
    print(one_grover_theory)
    print("Absolute error after one Q:")
    print(abs(success_probability_AQ - one_grover_theory))
    print("\nRaw probability dictionary after A then Q:")
    print(prob_dict_AQ)


Price grid:
[439.48215271 457.96346145 476.44477019 494.92607893 513.40738767
 531.88869641 550.37000515 568.85131389]

Payoff list:
[ 0.          0.          0.          0.         13.40738767 31.88869641
 50.37000515 68.85131389]

Gamma angles:
[0.         0.         0.         0.         0.63466491 1.00410095
 1.29904026 1.57079633]

Delta angles:
[ 0.56357531 -0.08014901 -0.01221    -0.15388384  0.15388384  0.01221
  0.08014901 -0.56357531]

Initial success probability P(q[3]=1) after A:
0.07076767299760532
Classical theory for A:
0.07076767299760531

Success probability P(q[3]=1) after A followed by one Q:
0.5223860760314368
Amplitude-amplification theory sin^2(3 theta):
0.5223860760314372
Absolute error after one Q:
3.3306690738754696e-16

Raw probability dictionary after A then Q:
{'0000': 0.0017796825563674242, '0001': 0.13388306899267752, '0010': 0.09048233380711505, '0011': 0.009901865633613964, '0100': 0.02051292856663429, '0101': 0.05026776820453747, '0110': 0.1697141572365

In [6]:
r"""# End-to-End SPY Option Pricing via IQAE

This script implements an iterative quantum amplitude estimation style workflow
on top of the manually compiled operators from the previous stages.

Let the success probability after the state-loading operator $\mathcal{A}$ be

\[
a=\sin^2(\theta).
\]

After applying the Grover iterate $\mathcal{Q}$ exactly $k$ times, the ideal
success probability becomes

\[
p_k(\theta)=\sin^2((2k+1)\theta).
\]

For each chosen Grover depth $k$, we perform a finite-shot Bernoulli sampling
experiment and observe $h_k$ successes out of $N_k$ shots. Under the binomial
model, the likelihood of $\theta$ is

\[
L(\theta)\propto \prod_k p_k(\theta)^{h_k}\left(1-p_k(\theta)\right)^{N_k-h_k}.
\]

Equivalently, we maximize the log-likelihood

\[
\log L(\theta)=\sum_k
\left[
h_k\log p_k(\theta)+(N_k-h_k)\log(1-p_k(\theta))
\right],
\qquad \theta\in[0,\pi/4].
\]

Once the maximum-likelihood estimator $\theta_{\mathrm{est}}$ is obtained, the
expected payoff is reconstructed by

\[
\mathbb{E}[\mathrm{payoff}]
=
\sin^2(\theta_{\mathrm{est}})\cdot \frac{\max(\mathrm{payoff})}{c},
\]

and the option price is discounted as

\[
V_0=e^{-rT}\,\mathbb{E}[\mathrm{payoff}].
\]

The script below uses the exact simulator probability to emulate NISQ sampling
through `numpy.random.binomial`, and then performs one-dimensional MLE either
with SciPy (if available) or with a dense grid-search fallback.
"""

from __future__ import annotations

import importlib.util
import math
from pathlib import Path

import numpy as np

try:
    from scipy.optimize import minimize_scalar
except Exception:  # pragma: no cover - grid-search fallback remains available
    minimize_scalar = None


def _load_grover_module():
    """Load the manually compiled Grover-operator module from the local file."""
    module_path = Path(__file__).with_name("04_SPY_Grover_Operator.py")
    spec = importlib.util.spec_from_file_location("spy_grover_operator", module_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Unable to load module from {module_path}.")

    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


core = _load_grover_module()

IQAE_CONTEXT: dict[str, object] = {}
SUCCESS_PROBABILITY_CACHE: dict[int, float] = {}
RNG = np.random.default_rng(20260327)


def initialize_iqae_context(
    strike: float = 500.0,
    scaling_constant: float = 0.5,
    s0: float = 500.0,
    r: float = 0.05,
    sigma: float = 0.15,
    maturity: float = 30.0 / 365.0,
    seed: int = 20260327,
) -> dict[str, object]:
    """Prepare all classical inputs and cached circuit angles for IQAE."""
    global IQAE_CONTEXT, SUCCESS_PROBABILITY_CACHE, RNG

    distribution = core.discretize_bsm_price_distribution(
        s0=s0,
        r=r,
        sigma=sigma,
        maturity=maturity,
        num_qubits=3,
        truncation_sigma=3.0,
    )

    price_grid = distribution["price_grid"]
    price_probabilities = distribution["probabilities"]
    price_amplitudes = distribution["amplitudes"]

    sp_angles = core.calculate_ry_angles(price_amplitudes)

    payoff_list = core.calculate_call_payoffs(price_grid, strike=strike)
    gamma_list, max_payoff, scaled_payoffs = core.calculate_gamma_angles(
        payoff_list,
        scaling_constant=scaling_constant,
    )
    delta_angles = core.calculate_payoff_angles(gamma_list)

    exact_a = float(np.sum(price_probabilities * scaled_payoffs))
    classical_grid_price = math.exp(-r * maturity) * float(np.sum(price_probabilities * payoff_list))

    IQAE_CONTEXT = {
        "strike": strike,
        "scaling_constant": scaling_constant,
        "s0": s0,
        "r": r,
        "sigma": sigma,
        "maturity": maturity,
        "price_grid": price_grid,
        "price_probabilities": price_probabilities,
        "price_amplitudes": price_amplitudes,
        "payoff_list": payoff_list,
        "max_payoff": max_payoff,
        "scaled_payoffs": scaled_payoffs,
        "gamma_list": gamma_list,
        "delta_angles": delta_angles,
        "sp_angles": sp_angles,
        "exact_a": exact_a,
        "theta_true": math.asin(math.sqrt(exact_a)),
        "classical_grid_price": classical_grid_price,
    }

    SUCCESS_PROBABILITY_CACHE = {}
    RNG = np.random.default_rng(seed)
    return IQAE_CONTEXT


def exact_success_probability(k: int) -> float:
    """Compute the exact success probability after A Q^k using the simulator."""
    if k < 0:
        raise ValueError("k must be nonnegative.")

    if k in SUCCESS_PROBABILITY_CACHE:
        return SUCCESS_PROBABILITY_CACHE[k]

    if not IQAE_CONTEXT:
        raise RuntimeError("IQAE context has not been initialized.")

    machine, q = core.create_machine_and_qubits(4)

    prog = core.QP.QProg()
    prog << core.build_A_circuit(
        IQAE_CONTEXT["sp_angles"],
        IQAE_CONTEXT["delta_angles"],
        q[:3],
        q[3],
    )

    for _ in range(k):
        prog << core.build_grover_Q(
            IQAE_CONTEXT["sp_angles"],
            IQAE_CONTEXT["delta_angles"],
            q[:3],
            q[3],
        )

    prob_dict = core.run_probabilities(machine, prog, q)
    success_probability = core.total_probability_target_one(prob_dict)

    SUCCESS_PROBABILITY_CACHE[k] = success_probability
    return success_probability


def run_and_sample(k: int, num_shots: int = 1000) -> tuple[int, float]:
    """Run A Q^k exactly, then emulate finite-shot NISQ sampling via binomial draws."""
    if num_shots <= 0:
        raise ValueError("num_shots must be positive.")

    success_probability = exact_success_probability(k)
    h_k = int(RNG.binomial(num_shots, success_probability))
    return h_k, success_probability


def collect_iqae_data(k_schedule: list[int], num_shots: int = 1000) -> list[dict[str, float | int]]:
    """Collect all (k, h_k, N_k) data records for the chosen IQAE schedule."""
    records = []
    for k in k_schedule:
        h_k, success_probability = run_and_sample(k, num_shots=num_shots)
        records.append(
            {
                "k": int(k),
                "shots": int(num_shots),
                "successes": int(h_k),
                "exact_probability": float(success_probability),
                "sample_frequency": float(h_k / num_shots),
            }
        )
    return records


def negative_log_likelihood(theta: float, data_records: list[dict[str, float | int]]) -> float:
    """Return the negative log-likelihood for the IQAE binomial model."""
    epsilon = 1e-12
    total = 0.0

    for record in data_records:
        k = int(record["k"])
        shots = int(record["shots"])
        successes = int(record["successes"])

        probability = math.sin((2 * k + 1) * theta) ** 2
        probability = min(max(probability, epsilon), 1.0 - epsilon)
        total -= successes * math.log(probability) + (shots - successes) * math.log(1.0 - probability)

    return total


def estimate_theta_mle(
    data_records: list[dict[str, float | int]],
    theta_min: float = 0.0,
    theta_max: float = math.pi / 4.0,
    grid_size: int = 200_001,
) -> tuple[float, str]:
    """Estimate theta by maximizing the total likelihood over all k records."""
    epsilon = 1e-12
    theta_grid = np.linspace(theta_min + epsilon, theta_max - epsilon, grid_size)
    nll_grid = np.zeros_like(theta_grid)

    for record in data_records:
        k = int(record["k"])
        shots = int(record["shots"])
        successes = int(record["successes"])

        probabilities = np.sin((2 * k + 1) * theta_grid) ** 2
        probabilities = np.clip(probabilities, epsilon, 1.0 - epsilon)
        nll_grid -= successes * np.log(probabilities) + (shots - successes) * np.log(1.0 - probabilities)

    best_index = int(np.argmin(nll_grid))
    theta_est = float(theta_grid[best_index])
    method = "grid-search"

    if minimize_scalar is not None:
        left_index = max(best_index - 1, 0)
        right_index = min(best_index + 1, theta_grid.size - 1)
        left = float(theta_grid[left_index])
        right = float(theta_grid[right_index])

        result = minimize_scalar(
            negative_log_likelihood,
            bounds=(left, right),
            method="bounded",
            args=(data_records,),
        )
        if result.success:
            theta_est = float(result.x)
            method = "scipy-bounded-mle"

    return theta_est, method


def price_from_theta(theta_est: float) -> float:
    """Reconstruct the discounted option price from the estimated amplitude angle."""
    if not IQAE_CONTEXT:
        raise RuntimeError("IQAE context has not been initialized.")

    scaling_constant = float(IQAE_CONTEXT["scaling_constant"])
    max_payoff = float(IQAE_CONTEXT["max_payoff"])
    r = float(IQAE_CONTEXT["r"])
    maturity = float(IQAE_CONTEXT["maturity"])

    expected_payoff_est = (math.sin(theta_est) ** 2) * max_payoff / scaling_constant
    return math.exp(-r * maturity) * expected_payoff_est


if __name__ == "__main__":
    np.set_printoptions(precision=8, suppress=True)

    k_schedule = [0, 1, 2, 4, 8]
    num_shots = 1000

    context = initialize_iqae_context()
    data_records = collect_iqae_data(k_schedule, num_shots=num_shots)
    theta_est, mle_method = estimate_theta_mle(data_records)

    theta_true = float(context["theta_true"])
    exact_a = float(context["exact_a"])
    classical_grid_price = float(context["classical_grid_price"])
    quantum_price = price_from_theta(theta_est)
    price_abs_error = abs(quantum_price - classical_grid_price)

    print("Price grid:")
    print(context["price_grid"])
    print("\nPayoff list:")
    print(context["payoff_list"])
    print("\nTrue amplitude a = sin^2(theta):")
    print(exact_a)
    print("True theta:")
    print(theta_true)
    print("\nIQAE schedule:")
    print(k_schedule)
    print("Shots per k:")
    print(num_shots)
    print("\nData records:")
    for record in data_records:
        print(record)
    print("\nMLE method:")
    print(mle_method)
    print("Estimated theta:")
    print(theta_est)
    print("Absolute theta error:")
    print(abs(theta_est - theta_true))
    print("\nQuantum Price:")
    print(quantum_price)
    print("Classical Grid Price:")
    print(classical_grid_price)
    print("Absolute pricing error:")
    print(price_abs_error)


NameError: name '__file__' is not defined

In [7]:
r"""# End-to-End SPY Option Pricing via IQAE

This script implements an iterative quantum amplitude estimation style workflow
on top of the manually compiled operators from the previous stages.

Let the success probability after the state-loading operator $\mathcal{A}$ be

\[
a=\sin^2(\theta).
\]

After applying the Grover iterate $\mathcal{Q}$ exactly $k$ times, the ideal
success probability becomes

\[
p_k(\theta)=\sin^2((2k+1)\theta).
\]

For each chosen Grover depth $k$, we perform a finite-shot Bernoulli sampling
experiment and observe $h_k$ successes out of $N_k$ shots. Under the binomial
model, the likelihood of $\theta$ is

\[
L(\theta)\propto \prod_k p_k(\theta)^{h_k}\left(1-p_k(\theta)\right)^{N_k-h_k}.
\]

Equivalently, we maximize the log-likelihood

\[
\log L(\theta)=\sum_k
\left[
h_k\log p_k(\theta)+(N_k-h_k)\log(1-p_k(\theta))
\right],
\qquad \theta\in[0,\pi/4].
\]

Once the maximum-likelihood estimator $\theta_{\mathrm{est}}$ is obtained, the
expected payoff is reconstructed by

\[
\mathbb{E}[\mathrm{payoff}]
=
\sin^2(\theta_{\mathrm{est}})\cdot \frac{\max(\mathrm{payoff})}{c},
\]

and the option price is discounted as

\[
V_0=e^{-rT}\,\mathbb{E}[\mathrm{payoff}].
\]

The script below uses the exact simulator probability to emulate NISQ sampling
through `numpy.random.binomial`, and then performs one-dimensional MLE either
with SciPy (if available) or with a dense grid-search fallback.
"""

from __future__ import annotations

import importlib.util
import math
from pathlib import Path
import types

import numpy as np

try:
    from scipy.optimize import minimize_scalar
except Exception:  # pragma: no cover - grid-search fallback remains available
    minimize_scalar = None


def _load_grover_module():
    """Load the manually compiled Grover-operator module from the local file.

    This helper supports both:
    - normal `.py` execution, where `__file__` exists
    - Jupyter notebooks, where `__file__` is usually undefined
    """
    required_names = [
        "QP",
        "discretize_bsm_price_distribution",
        "calculate_ry_angles",
        "calculate_call_payoffs",
        "calculate_gamma_angles",
        "calculate_payoff_angles",
        "create_machine_and_qubits",
        "build_A_circuit",
        "build_grover_Q",
        "run_probabilities",
        "total_probability_target_one",
    ]

    if all(name in globals() for name in required_names):
        return types.SimpleNamespace(**{name: globals()[name] for name in required_names})

    candidate_paths = []
    if "__file__" in globals():
        candidate_paths.append(Path(__file__).resolve().with_name("04_SPY_Grover_Operator.py"))
    candidate_paths.append((Path.cwd() / "04_SPY_Grover_Operator.py").resolve())

    checked_paths = []
    seen_paths = set()

    for module_path in candidate_paths:
        module_path_str = str(module_path)
        if module_path_str in seen_paths:
            continue
        seen_paths.add(module_path_str)
        checked_paths.append(module_path_str)

        if not module_path.exists():
            continue

        spec = importlib.util.spec_from_file_location("spy_grover_operator", module_path)
        if spec is None or spec.loader is None:
            raise ImportError(f"Unable to load module from {module_path}.")

        module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(module)
        return module

    raise FileNotFoundError(
        "Unable to locate `04_SPY_Grover_Operator.py`. "
        "Please keep it in the current working directory or the same directory as this script. "
        f"Searched paths: {checked_paths}"
    )


core = _load_grover_module()

IQAE_CONTEXT: dict[str, object] = {}
SUCCESS_PROBABILITY_CACHE: dict[int, float] = {}
RNG = np.random.default_rng(20260327)


def initialize_iqae_context(
    strike: float = 500.0,
    scaling_constant: float = 0.5,
    s0: float = 500.0,
    r: float = 0.05,
    sigma: float = 0.15,
    maturity: float = 30.0 / 365.0,
    seed: int = 20260327,
) -> dict[str, object]:
    """Prepare all classical inputs and cached circuit angles for IQAE."""
    global IQAE_CONTEXT, SUCCESS_PROBABILITY_CACHE, RNG

    distribution = core.discretize_bsm_price_distribution(
        s0=s0,
        r=r,
        sigma=sigma,
        maturity=maturity,
        num_qubits=3,
        truncation_sigma=3.0,
    )

    price_grid = distribution["price_grid"]
    price_probabilities = distribution["probabilities"]
    price_amplitudes = distribution["amplitudes"]

    sp_angles = core.calculate_ry_angles(price_amplitudes)

    payoff_list = core.calculate_call_payoffs(price_grid, strike=strike)
    gamma_list, max_payoff, scaled_payoffs = core.calculate_gamma_angles(
        payoff_list,
        scaling_constant=scaling_constant,
    )
    delta_angles = core.calculate_payoff_angles(gamma_list)

    exact_a = float(np.sum(price_probabilities * scaled_payoffs))
    classical_grid_price = math.exp(-r * maturity) * float(np.sum(price_probabilities * payoff_list))

    IQAE_CONTEXT = {
        "strike": strike,
        "scaling_constant": scaling_constant,
        "s0": s0,
        "r": r,
        "sigma": sigma,
        "maturity": maturity,
        "price_grid": price_grid,
        "price_probabilities": price_probabilities,
        "price_amplitudes": price_amplitudes,
        "payoff_list": payoff_list,
        "max_payoff": max_payoff,
        "scaled_payoffs": scaled_payoffs,
        "gamma_list": gamma_list,
        "delta_angles": delta_angles,
        "sp_angles": sp_angles,
        "exact_a": exact_a,
        "theta_true": math.asin(math.sqrt(exact_a)),
        "classical_grid_price": classical_grid_price,
    }

    SUCCESS_PROBABILITY_CACHE = {}
    RNG = np.random.default_rng(seed)
    return IQAE_CONTEXT


def exact_success_probability(k: int) -> float:
    """Compute the exact success probability after A Q^k using the simulator."""
    if k < 0:
        raise ValueError("k must be nonnegative.")

    if k in SUCCESS_PROBABILITY_CACHE:
        return SUCCESS_PROBABILITY_CACHE[k]

    if not IQAE_CONTEXT:
        raise RuntimeError("IQAE context has not been initialized.")

    machine, q = core.create_machine_and_qubits(4)

    prog = core.QP.QProg()
    prog << core.build_A_circuit(
        IQAE_CONTEXT["sp_angles"],
        IQAE_CONTEXT["delta_angles"],
        q[:3],
        q[3],
    )

    for _ in range(k):
        prog << core.build_grover_Q(
            IQAE_CONTEXT["sp_angles"],
            IQAE_CONTEXT["delta_angles"],
            q[:3],
            q[3],
        )

    prob_dict = core.run_probabilities(machine, prog, q)
    success_probability = core.total_probability_target_one(prob_dict)

    SUCCESS_PROBABILITY_CACHE[k] = success_probability
    return success_probability


def run_and_sample(k: int, num_shots: int = 1000) -> tuple[int, float]:
    """Run A Q^k exactly, then emulate finite-shot NISQ sampling via binomial draws."""
    if num_shots <= 0:
        raise ValueError("num_shots must be positive.")

    success_probability = exact_success_probability(k)
    h_k = int(RNG.binomial(num_shots, success_probability))
    return h_k, success_probability


def collect_iqae_data(k_schedule: list[int], num_shots: int = 1000) -> list[dict[str, float | int]]:
    """Collect all (k, h_k, N_k) data records for the chosen IQAE schedule."""
    records = []
    for k in k_schedule:
        h_k, success_probability = run_and_sample(k, num_shots=num_shots)
        records.append(
            {
                "k": int(k),
                "shots": int(num_shots),
                "successes": int(h_k),
                "exact_probability": float(success_probability),
                "sample_frequency": float(h_k / num_shots),
            }
        )
    return records


def negative_log_likelihood(theta: float, data_records: list[dict[str, float | int]]) -> float:
    """Return the negative log-likelihood for the IQAE binomial model."""
    epsilon = 1e-12
    total = 0.0

    for record in data_records:
        k = int(record["k"])
        shots = int(record["shots"])
        successes = int(record["successes"])

        probability = math.sin((2 * k + 1) * theta) ** 2
        probability = min(max(probability, epsilon), 1.0 - epsilon)
        total -= successes * math.log(probability) + (shots - successes) * math.log(1.0 - probability)

    return total


def estimate_theta_mle(
    data_records: list[dict[str, float | int]],
    theta_min: float = 0.0,
    theta_max: float = math.pi / 4.0,
    grid_size: int = 200_001,
) -> tuple[float, str]:
    """Estimate theta by maximizing the total likelihood over all k records."""
    epsilon = 1e-12
    theta_grid = np.linspace(theta_min + epsilon, theta_max - epsilon, grid_size)
    nll_grid = np.zeros_like(theta_grid)

    for record in data_records:
        k = int(record["k"])
        shots = int(record["shots"])
        successes = int(record["successes"])

        probabilities = np.sin((2 * k + 1) * theta_grid) ** 2
        probabilities = np.clip(probabilities, epsilon, 1.0 - epsilon)
        nll_grid -= successes * np.log(probabilities) + (shots - successes) * np.log(1.0 - probabilities)

    best_index = int(np.argmin(nll_grid))
    theta_est = float(theta_grid[best_index])
    method = "grid-search"

    if minimize_scalar is not None:
        left_index = max(best_index - 1, 0)
        right_index = min(best_index + 1, theta_grid.size - 1)
        left = float(theta_grid[left_index])
        right = float(theta_grid[right_index])

        result = minimize_scalar(
            negative_log_likelihood,
            bounds=(left, right),
            method="bounded",
            args=(data_records,),
        )
        if result.success:
            theta_est = float(result.x)
            method = "scipy-bounded-mle"

    return theta_est, method


def price_from_theta(theta_est: float) -> float:
    """Reconstruct the discounted option price from the estimated amplitude angle."""
    if not IQAE_CONTEXT:
        raise RuntimeError("IQAE context has not been initialized.")

    scaling_constant = float(IQAE_CONTEXT["scaling_constant"])
    max_payoff = float(IQAE_CONTEXT["max_payoff"])
    r = float(IQAE_CONTEXT["r"])
    maturity = float(IQAE_CONTEXT["maturity"])

    expected_payoff_est = (math.sin(theta_est) ** 2) * max_payoff / scaling_constant
    return math.exp(-r * maturity) * expected_payoff_est


if __name__ == "__main__":
    np.set_printoptions(precision=8, suppress=True)

    k_schedule = [0, 1, 2, 4, 8]
    num_shots = 1000

    context = initialize_iqae_context()
    data_records = collect_iqae_data(k_schedule, num_shots=num_shots)
    theta_est, mle_method = estimate_theta_mle(data_records)

    theta_true = float(context["theta_true"])
    exact_a = float(context["exact_a"])
    classical_grid_price = float(context["classical_grid_price"])
    quantum_price = price_from_theta(theta_est)
    price_abs_error = abs(quantum_price - classical_grid_price)

    print("Price grid:")
    print(context["price_grid"])
    print("\nPayoff list:")
    print(context["payoff_list"])
    print("\nTrue amplitude a = sin^2(theta):")
    print(exact_a)
    print("True theta:")
    print(theta_true)
    print("\nIQAE schedule:")
    print(k_schedule)
    print("Shots per k:")
    print(num_shots)
    print("\nData records:")
    for record in data_records:
        print(record)
    print("\nMLE method:")
    print(mle_method)
    print("Estimated theta:")
    print(theta_est)
    print("Absolute theta error:")
    print(abs(theta_est - theta_true))
    print("\nQuantum Price:")
    print(quantum_price)
    print("Classical Grid Price:")
    print(classical_grid_price)
    print("Absolute pricing error:")
    print(price_abs_error)


Price grid:
[439.48215271 457.96346145 476.44477019 494.92607893 513.40738767
 531.88869641 550.37000515 568.85131389]

Payoff list:
[ 0.          0.          0.          0.         13.40738767 31.88869641
 50.37000515 68.85131389]

True amplitude a = sin^2(theta):
0.07076767299760531
True theta:
0.26926390838945574

IQAE schedule:
[0, 1, 2, 4, 8]
Shots per k:
1000

Data records:
{'k': 0, 'shots': 1000, 'successes': 75, 'exact_probability': 0.07076767299760532, 'sample_frequency': 0.075}
{'k': 1, 'shots': 1000, 'successes': 502, 'exact_probability': 0.5223860760314368, 'sample_frequency': 0.502}
{'k': 2, 'shots': 1000, 'successes': 958, 'exact_probability': 0.9504508851919551, 'sample_frequency': 0.958}
{'k': 4, 'shots': 1000, 'successes': 414, 'exact_probability': 0.43302126754655795, 'sample_frequency': 0.414}
{'k': 8, 'shots': 1000, 'successes': 989, 'exact_probability': 0.9819114352702701, 'sample_frequency': 0.989}

MLE method:
scipy-bounded-mle
Estimated theta:
0.2710321251703554